In [1]:
import sys
sys.path.append(r'../')
import numpy as np
import matplotlib.pyplot as plt


from src.d_CSL import GcStar
from src.d_CSL import Visualize_on_topography
from src.dataload import *
from src.dataPreProcessing import replace_nan

# if variables are changed on the fly 
from reloading import reloading
from tqdm.notebook import tqdm

## Analysis

In [ ]:
# %%time
metrics, N_PASTS, num = [], list(range(1, 8)), 20
for a in tqdm(range(num)):
    n_neur = np.random.randint(25, 40, 1)[0] 
    l = np.random.randint(5000, 15000, 1)[0]
    alpha, beta, n_lags = 0.01, 0.001, 1
    
    
    # simulate data
    
    A = adj_mtx(n_neur)
    noise = continuous_noise_fun(num = n_neur, l = l)
    X = np.zeros((A.shape[0], l)).T
    X[0] = np.random.randn(A.shape[0])
    for i, row in enumerate(X[:-1]):
        X[i+1] = A @ X[i] + np.random.normal(0, 0.25, A.shape[0]) + noise[:, i]
    X=X.T
    
    print(f'Data {a} of {num} with {X.shape[0]} vars')
    
    # fit algorithm to data
    met = np.zeros((len(N_PASTS),4))
    for k, n_past in enumerate(N_PASTS):
        print(n_past)
        
        # intantiate the GcStar()
        gcstar = GcStar(n_perm = 1000, n_pasts = n_past, 
                        n_lags = n_lags, temporal = True, method = ' ')
        
        gcstar.fit(X, verbose=0)
        
        # compute metrics
        gcstar.get_connectivity_matrix()
        gcstar.compute_confusion_matrix(A, simulation = True)
        met[k] = gcstar.compute_metrics()
    metrics.append(met)

  0%|          | 0/20 [00:00<?, ?it/s]

Data 0 of 20 with 33 vars
1
2
3
4
5
6
7
Data 1 of 20 with 38 vars
1
2
3
4
5
6
7
Data 2 of 20 with 36 vars
1
2
3
4
5
6
7
Data 3 of 20 with 36 vars
1
2
3
4
5
6
7
Data 4 of 20 with 38 vars
1
2
3
4
5
6
7
Data 5 of 20 with 31 vars
1
2
3
4
5
6
7
Data 6 of 20 with 32 vars
1
2
3
4
5
6
7
Data 7 of 20 with 35 vars
1
2
3
4
5
6
7
Data 8 of 20 with 39 vars
1
2
3
4
5
6
7
Data 9 of 20 with 25 vars
1
2
3
4
5
6
7
Data 10 of 20 with 28 vars
1
2
3
4
5
6
7
Data 11 of 20 with 36 vars
1
2
3
4
5
6
7
Data 12 of 20 with 35 vars
1
2
3
4
5
6
7


In [ ]:
for met in metrics:
    print(met)

In [ ]:
Metrics = np.zeros((num,7,4))
for i, met in enumerate(metrics):
    Metrics[i] = met
np.mean(Metrics[:,0,0])

In [ ]:
np.mean(Metrics,axis=0)

In [ ]:
fig, ax = plt.subplots(1, 4, figsize = (12, 3))
labels = ['Accuracy', 'Precision', 'Recall', 'FPR']
lab = ['$\mu_{acc}$','$\mu_{prec}$', '$\mu_{rec}$', '$\mu_{FPR}$']
for i in range(Metrics.shape[2]):
    ax[i].plot(range(1, 1 + Metrics.shape[1]), 
               np.mean(Metrics, axis = 0)[:, i],
               marker = '.', label = lab[i])
    
    ax[i].errorbar(range(1, 1 + Metrics.shape[1]), 
                   np.mean(Metrics, axis = 0)[:, i], 
                   yerr = np.std(Metrics, axis = 0)[:, i], 
                   fmt='o')
    ax[i].set_xticks(range(1,1+Metrics.shape[1]), 
                     list(range(1, 1+Metrics.shape[1])))
    ax[i].set_xlabel('$n_{pasts}$')
    ax[i].set_ylabel('%', rotation = 180)
    ax[i].set_title(f'{labels[i]}')
    ax[i].legend()
    plt.tight_layout()